# Elliptic Bitcoin — Exploratory Data Analysis

Dataset: 203k transactions on the Bitcoin blockchain.
- **Class 1** = illicit (anomaly)
- **Class 2** = licit (normal)
- **unknown** = unlabeled


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_all
from src.preprocessing import get_labeled, get_feature_cols

sns.set_theme(style='whitegrid')

df, edges = load_all()
print(df.shape)
df.head()

## Class Distribution

In [ ]:
counts = df['class'].value_counts()
print(counts)

labeled = df[df['class'].isin(['1','2'])]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts.plot(kind='bar', ax=axes[0], color=['#e74c3c','#2ecc71','#95a5a6'])
axes[0].set_title('All transactions')
axes[0].set_xticklabels(['unknown','licit (2)','illicit (1)'], rotation=0)

labeled['class'].value_counts().plot(kind='bar', ax=axes[1], color=['#2ecc71','#e74c3c'])
axes[1].set_title('Labeled only')
axes[1].set_xticklabels(['licit (2)','illicit (1)'], rotation=0)

plt.tight_layout()
plt.show()

illicit_pct = (labeled['class']=='1').mean() * 100
print(f'Illicit rate among labeled: {illicit_pct:.1f}%')

## Class Distribution Over Time Steps

In [ ]:
time_class = df.groupby(['time_step', 'class']).size().unstack(fill_value=0)
time_class.plot(figsize=(14, 5), title='Transactions per time step by class')
plt.xlabel('Time step')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## Feature Statistics

In [ ]:
labeled_df = get_labeled(df)
feat_cols = get_feature_cols(labeled_df)
print(f'Feature count: {len(feat_cols)}')

X_illicit = labeled_df[labeled_df['label']==1][feat_cols]
X_licit   = labeled_df[labeled_df['label']==0][feat_cols]

print('\nIllicit stats (first 5 features):')
print(X_illicit[feat_cols[:5]].describe().round(3))

## Feature Distributions — Illicit vs Licit (sample)

In [ ]:
sample_feats = feat_cols[1:7]
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.ravel()

for i, col in enumerate(sample_feats):
    axes[i].hist(X_licit[col].clip(-5, 5), bins=60, alpha=0.6, label='licit', color='#2ecc71', density=True)
    axes[i].hist(X_illicit[col].clip(-5, 5), bins=60, alpha=0.6, label='illicit', color='#e74c3c', density=True)
    axes[i].set_title(col)
    axes[i].legend(fontsize=8)

plt.suptitle('Local features: illicit vs licit (clipped at ±5σ)', y=1.02)
plt.tight_layout()
plt.show()

## Correlation Heatmap (first 20 local features)

In [ ]:
lf_cols = [c for c in feat_cols if c.startswith('lf_')][:20]
corr = labeled_df[lf_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, cmap='coolwarm', center=0, ax=ax, linewidths=0.3)
ax.set_title('Correlation — first 20 local features')
plt.tight_layout()
plt.show()

## Graph Stats

In [ ]:
print(f'Nodes: {df["txId"].nunique():,}')
print(f'Edges: {len(edges):,}')
print(f'Avg degree: {len(edges)*2 / df["txId"].nunique():.2f}')

degree = pd.concat([edges['txId1'], edges['txId2']]).value_counts()
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(degree.values, bins=50, log=True, color='steelblue')
ax.set_xlabel('Node degree')
ax.set_ylabel('Count (log)')
ax.set_title('Degree distribution')
plt.tight_layout()
plt.show()